## 🎯 Learning Objectives
* Understand the components of end-to-end RAG pipeline evaluation.
* Implement key retrieval metrics such as Hit Rate, MRR, Recall@k, and Precision@k.
* Implement generation and overall RAG metrics, including Faithfulness, Answer Relevancy, Context Relevancy, and Answer Correctness, leveraging LLM-as-a-judge techniques.
* Apply these evaluation techniques to a LlamaIndex-based RAG pipeline.
* Interpret evaluation results to identify strengths and weaknesses in the RAG system and guide iterative improvements.


## Exercise: End-to-End Evaluation of a Production RAG Pipeline

### Context
In the journey of building production-ready RAG systems, deployment is only half the battle. Continuous evaluation is paramount to ensure the system consistently delivers accurate, relevant, and faithful responses. Without a robust evaluation framework, it's impossible to measure the impact of changes, identify regressions, or confidently iterate on improvements. This exercise focuses on establishing a comprehensive end-to-end evaluation pipeline for a LlamaIndex-powered RAG system.

### Task
Your task is to implement an end-to-end evaluation framework for a given LlamaIndex RAG pipeline. This involves defining a suitable evaluation dataset, implementing various retrieval and generation metrics, and then running a full evaluation against a mock RAG setup. You will need to collect results, calculate metrics, and present them clearly.

### Requirements
1.  **Evaluation Dataset:** Create a small, representative evaluation dataset consisting of:
    *   `queries`: A list of user questions.
    *   `ground_truth_contexts`: For each query, a list of text snippets that are considered relevant ground-truth contexts.
    *   `ground_truth_answers`: For each query, the expected correct answer.

2.  **Retrieval Metrics Implementation:** Implement functions to calculate the following retrieval metrics:
    *   **Hit Rate:** The proportion of queries for which the ground-truth context is found within the top-k retrieved documents.
    *   **Mean Reciprocal Rank (MRR):** A measure of the average reciprocal rank of the first relevant document.
    *   **Recall@k:** The proportion of relevant documents retrieved within the top-k results.
    *   **Precision@k:** The proportion of retrieved documents that are relevant within the top-k results.

3.  **Generation & Overall RAG Metrics:** Utilize modern LLM-as-a-judge frameworks (like LlamaIndex's built-in evaluators or `ragas`) to assess the quality of generated answers. Implement or integrate metrics for:
    *   **Faithfulness:** Measures if the generated answer is grounded in the retrieved context.
    *   **Answer Relevancy:** Measures if the generated answer directly addresses the user's query.
    *   **Context Relevancy:** Measures if the retrieved context is relevant to the user's query.
    *   **Answer Correctness:** Measures how factually accurate the generated answer is compared to the ground-truth answer.

4.  **Evaluation Loop:** Construct an evaluation loop that:
    *   Iterates through each query in your evaluation dataset.
    *   Executes the LlamaIndex `QueryEngine` to get retrieved nodes and a generated response.
    *   Collects necessary data (query, response, retrieved contexts, ground-truth contexts, ground-truth answer) for metric calculation.

5.  **Results Presentation:** Present the calculated metrics clearly, ideally in a summary table, to provide an overview of the RAG pipeline's performance.

### Evaluation Criteria
*   **Correctness:** Are the metric implementations accurate and do they produce expected results?
*   **Robustness:** Is the evaluation pipeline well-structured and capable of handling different types of queries and responses?
*   **Clarity:** Are the evaluation results easy to understand and interpret?
*   **Modernity:** Does the solution leverage modern LlamaIndex evaluation utilities and best practices (e.g., LLM-as-a-judge for generation metrics)?
*   **Code Quality:** Is the code clean, well-commented, and modular?


In [ ]:
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple

# LlamaIndex imports
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

# Ragas imports (for advanced generation metrics)
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_relevancy, answer_correctness
from datasets import Dataset

# --- Configuration and Setup ---

# Set your OpenAI API key. For production, use environment variables.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Mock API key for demonstration if not provided
if "OPENAI_API_KEY" not in os.environ:
    print("WARNING: OPENAI_API_KEY not found. Using a placeholder. Please set your key for actual runs.")
    os.environ["OPENAI_API_KEY"] = "sk-mock-key-for-exercise"

# Configure LlamaIndex settings for 2026 readiness
# Using modern OpenAI models and embedding models
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)

print(f"LlamaIndex LLM: {Settings.llm.model}")
print(f"LlamaIndex Embedding Model: {Settings.embed_model.model_name}")

# --- Mock Data Generation ---

# 1. Create a set of mock documents
raw_documents = [
    "The capital of France is Paris. Paris is famous for the Eiffel Tower and the Louvre Museum.",
    "Germany's capital is Berlin, a city rich in history, including the Brandenburg Gate and the Berlin Wall.",
    "The Amazon rainforest is the largest tropical rainforest in the world, spanning several South American countries.",
    "Artificial intelligence (AI) is rapidly transforming industries by enabling machines to learn and perform human-like tasks.",
    "Quantum computing harnesses quantum-mechanical phenomena like superposition and entanglement to solve problems intractable for classical computers.",
    "Renewable energy sources, such as solar and wind power, are crucial for combating climate change and ensuring sustainable development.",
    "The human brain is an incredibly complex organ, responsible for thought, memory, emotion, and every aspect of our lives."
]

docs = [Document(text=t) for t in raw_documents]

# 2. Build a mock LlamaIndex VectorStoreIndex and QueryEngine
print("Building mock LlamaIndex...")
index = VectorStoreIndex.from_documents(docs)
query_engine = index.as_query_engine(similarity_top_k=2)
print("Mock LlamaIndex built.")

# 3. Define a mock evaluation dataset
# This dataset includes queries, ground-truth relevant contexts, and ground-truth answers.
# In a real scenario, this would be a carefully curated dataset.

eval_data = [
    {
        "query": "What is the capital of France and what is it known for?",
        "ground_truth_contexts": [
            "The capital of France is Paris. Paris is famous for the Eiffel Tower and the Louvre Museum."
        ],
        "ground_truth_answer": "The capital of France is Paris, known for the Eiffel Tower and the Louvre Museum."
    },
    {
        "query": "Tell me about Germany's capital.",
        "ground_truth_contexts": [
            "Germany's capital is Berlin, a city rich in history, including the Brandenburg Gate and the Berlin Wall."
        ],
        "ground_truth_answer": "Germany's capital is Berlin, a historical city featuring landmarks like the Brandenburg Gate and the Berlin Wall."
    },
    {
        "query": "Which is the largest rainforest globally?",
        "ground_truth_contexts": [
            "The Amazon rainforest is the largest tropical rainforest in the world, spanning several South American countries."
        ],
        "ground_truth_answer": "The Amazon rainforest is the largest tropical rainforest in the world."
    },
    {
        "query": "How is AI transforming industries?",
        "ground_truth_contexts": [
            "Artificial intelligence (AI) is rapidly transforming industries by enabling machines to learn and perform human-like tasks."
        ],
        "ground_truth_answer": "AI is transforming industries by allowing machines to learn and perform human-like tasks."
    },
    {
        "query": "What is quantum computing?",
        "ground_truth_contexts": [
            "Quantum computing harnesses quantum-mechanical phenomena like superposition and entanglement to solve problems intractable for classical computers."
        ],
        "ground_truth_answer": "Quantum computing uses quantum mechanics (superposition, entanglement) to solve complex problems that classical computers cannot."
    },
    {
        "query": "Why are renewable energy sources important?",
        "ground_truth_contexts": [
            "Renewable energy sources, such as solar and wind power, are crucial for combating climate change and ensuring sustainable development."
        ],
        "ground_truth_answer": "Renewable energy sources are important for fighting climate change and achieving sustainable development."
    },
    {
        "query": "What is the function of the human brain?",
        "ground_truth_contexts": [
            "The human brain is an incredibly complex organ, responsible for thought, memory, emotion, and every aspect of our lives."
        ],
        "ground_truth_answer": "The human brain is a complex organ responsible for thought, memory, emotion, and all aspects of life."
    }
]

# Convert to DataFrame for easier processing
eval_df = pd.DataFrame(eval_data)

print(f"Evaluation dataset loaded with {len(eval_df)} samples.")
print("Setup complete. Proceed to implement the evaluation logic.")


### Your Implementation Here

Now it's your turn! Based on the requirements outlined in the task, implement the end-to-end evaluation pipeline. You should:

1.  **Define helper functions** for calculating retrieval metrics (Hit Rate, MRR, Recall@k, Precision@k).
2.  **Set up LlamaIndex evaluators** for Faithfulness and Relevancy, and potentially `ragas` for a broader set of generation/RAG metrics.
3.  **Implement the main evaluation loop** that iterates through the `eval_df` (from the setup code), queries the `query_engine`, and collects all necessary data.
4.  **Calculate and aggregate** all defined metrics.
5.  **Present the results** in a clear and concise manner.

Feel free to add any additional helper functions or data structures you deem necessary for a robust evaluation.


In [ ]:
import os
import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple

# LlamaIndex imports
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator

# Ragas imports (for advanced generation metrics)
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_relevancy, answer_correctness
from datasets import Dataset

# --- Configuration and Setup (repeated for self-contained solution) ---

# Set your OpenAI API key. For production, use environment variables.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Mock API key for demonstration if not provided
if "OPENAI_API_KEY" not in os.environ or os.environ["OPENAI_API_KEY"] == "sk-mock-key-for-exercise":
    print("WARNING: OPENAI_API_KEY not found or is a placeholder. Using a placeholder. Please set your key for actual runs.")
    os.environ["OPENAI_API_KEY"] = "sk-mock-key-for-exercise"

# Configure LlamaIndex settings for 2026 readiness
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)

# --- Mock Data Generation (repeated for self-contained solution) ---
raw_documents = [
    "The capital of France is Paris. Paris is famous for the Eiffel Tower and the Louvre Museum.",
    "Germany's capital is Berlin, a city rich in history, including the Brandenburg Gate and the Berlin Wall.",
    "The Amazon rainforest is the largest tropical rainforest in the world, spanning several South American countries.",
    "Artificial intelligence (AI) is rapidly transforming industries by enabling machines to learn and perform human-like tasks.",
    "Quantum computing harnesses quantum-mechanical phenomena like superposition and entanglement to solve problems intractable for classical computers.",
    "Renewable energy sources, such as solar and wind power, are crucial for combating climate change and ensuring sustainable development.",
    "The human brain is an incredibly complex organ, responsible for thought, memory, emotion, and every aspect of our lives."
]

docs = [Document(text=t) for t in raw_documents]

print("Building mock LlamaIndex...")
index = VectorStoreIndex.from_documents(docs)
query_engine = index.as_query_engine(similarity_top_k=2)
print("Mock LlamaIndex built.")

eval_data = [
    {
        "query": "What is the capital of France and what is it known for?",
        "ground_truth_contexts": [
            "The capital of France is Paris. Paris is famous for the Eiffel Tower and the Louvre Museum."
        ],
        "ground_truth_answer": "The capital of France is Paris, known for the Eiffel Tower and the Louvre Museum."
    },
    {
        "query": "Tell me about Germany's capital.",
        "ground_truth_contexts": [
            "Germany's capital is Berlin, a city rich in history, including the Brandenburg Gate and the Berlin Wall."
        ],
        "ground_truth_answer": "Germany's capital is Berlin, a historical city featuring landmarks like the Brandenburg Gate and the Berlin Wall."
    },
    {
        "query": "Which is the largest rainforest globally?",
        "ground_truth_contexts": [
            "The Amazon rainforest is the largest tropical rainforest in the world, spanning several South American countries."
        ],
        "ground_truth_answer": "The Amazon rainforest is the largest tropical rainforest in the world."
    },
    {
        "query": "How is AI transforming industries?",
        "ground_truth_contexts": [
            "Artificial intelligence (AI) is rapidly transforming industries by enabling machines to learn and perform human-like tasks."
        ],
        "ground_truth_answer": "AI is transforming industries by allowing machines to learn and perform human-like tasks."
    },
    {
        "query": "What is quantum computing?",
        "ground_truth_contexts": [
            "Quantum computing harnesses quantum-mechanical phenomena like superposition and entanglement to solve problems intractable for classical computers."
        ],
        "ground_truth_answer": "Quantum computing uses quantum mechanics (superposition, entanglement) to solve complex problems that classical computers cannot."
    },
    {
        "query": "Why are renewable energy sources important?",
        "ground_truth_contexts": [
            "Renewable energy sources, such as solar and wind power, are crucial for combating climate change and ensuring sustainable development."
        ],
        "ground_truth_answer": "Renewable energy sources are important for fighting climate change and achieving sustainable development."
    },
    {
        "query": "What is the function of the human brain?",
        "ground_truth_contexts": [
            "The human brain is an incredibly complex organ, responsible for thought, memory, emotion, and every aspect of our lives."
        ],
        "ground_truth_answer": "The human brain is a complex organ responsible for thought, memory, emotion, and all aspects of life."
    }
]

eval_df = pd.DataFrame(eval_data)

# --- Reference Solution --- 

# 1. Retrieval Metrics Implementation

def calculate_hit_rate(retrieved_nodes_list: List[List[str]], ground_truth_contexts_list: List[List[str]], k: int = 1) -> float:
    """Calculates the Hit Rate: proportion of queries where at least one ground-truth context is in top-k retrieved nodes."""
    hits = 0
    for i in range(len(retrieved_nodes_list)):
        retrieved_texts = [node_text.strip().lower() for node_text in retrieved_nodes_list[i][:k]]
        gt_contexts = [gt_text.strip().lower() for gt_text in ground_truth_contexts_list[i]]
        
        # Check if any ground truth context is present in the retrieved texts
        if any(any(gt_ctx in retrieved_text for retrieved_text in retrieved_texts) for gt_ctx in gt_contexts):
            hits += 1
    return hits / len(retrieved_nodes_list)

def calculate_mrr(retrieved_nodes_list: List[List[str]], ground_truth_contexts_list: List[List[str]]) -> float:
    """Calculates Mean Reciprocal Rank (MRR)."""
    reciprocal_ranks = []
    for i in range(len(retrieved_nodes_list)):
        retrieved_texts = [node_text.strip().lower() for node_text in retrieved_nodes_list[i]]
        gt_contexts = [gt_text.strip().lower() for gt_text in ground_truth_contexts_list[i]]
        
        found = False
        for rank, retrieved_text in enumerate(retrieved_texts):
            if any(gt_ctx in retrieved_text for gt_ctx in gt_contexts):
                reciprocal_ranks.append(1 / (rank + 1))
                found = True
                break
        if not found:
            reciprocal_ranks.append(0) # No relevant document found
    return np.mean(reciprocal_ranks)

def calculate_recall_at_k(retrieved_nodes_list: List[List[str]], ground_truth_contexts_list: List[List[str]], k: int = 1) -> float:
    """Calculates Recall@k: proportion of relevant documents retrieved within top-k."""
    total_relevant_found = 0
    total_relevant_docs = 0
    for i in range(len(retrieved_nodes_list)):
        retrieved_texts = [node_text.strip().lower() for node_text in retrieved_nodes_list[i][:k]]
        gt_contexts = [gt_text.strip().lower() for gt_text in ground_truth_contexts_list[i]]
        
        total_relevant_docs += len(gt_contexts)
        
        for gt_ctx in gt_contexts:
            if any(gt_ctx in retrieved_text for retrieved_text in retrieved_texts):
                total_relevant_found += 1
    
    return total_relevant_found / total_relevant_docs if total_relevant_docs > 0 else 0.0

def calculate_precision_at_k(retrieved_nodes_list: List[List[str]], ground_truth_contexts_list: List[List[str]], k: int = 1) -> float:
    """Calculates Precision@k: proportion of retrieved documents that are relevant within top-k."""
    total_precision = 0
    for i in range(len(retrieved_nodes_list)):
        retrieved_texts = [node_text.strip().lower() for node_text in retrieved_nodes_list[i][:k]]
        gt_contexts = [gt_text.strip().lower() for gt_text in ground_truth_contexts_list[i]]
        
        if not retrieved_texts: # Avoid division by zero if no nodes retrieved
            continue
            
        relevant_retrieved_count = 0
        for retrieved_text in retrieved_texts:
            if any(gt_ctx in retrieved_text for gt_ctx in gt_contexts):
                relevant_retrieved_count += 1
        
        total_precision += (relevant_retrieved_count / len(retrieved_texts))
        
    return total_precision / len(retrieved_nodes_list) if len(retrieved_nodes_list) > 0 else 0.0


# 2. Setup LlamaIndex and Ragas Evaluators

# LlamaIndex evaluators
faithfulness_evaluator = FaithfulnessEvaluator(llm=Settings.llm)
relevancy_evaluator = RelevancyEvaluator(llm=Settings.llm)

# Ragas metrics (using default LLM from Settings.llm)
ragas_metrics = [
    faithfulness,
    answer_relevancy,
    context_relevancy,
    answer_correctness
]

# 3. Main Evaluation Loop

results = []
ragas_data = [] # To store data for Ragas evaluation

print("\nStarting end-to-end RAG evaluation...")
for idx, row in eval_df.iterrows():
    query = row["query"]
    ground_truth_contexts = row["ground_truth_contexts"]
    ground_truth_answer = row["ground_truth_answer"]

    print(f"\nProcessing query {idx+1}/{len(eval_df)}: {query}")
    
    # Execute RAG pipeline
    response = query_engine.query(query)
    
    retrieved_nodes_text = [node.get_content() for node in response.source_nodes]
    generated_answer = response.response
    
    # Store results for retrieval metrics
    results.append({
        "query": query,
        "retrieved_nodes": retrieved_nodes_text,
        "ground_truth_contexts": ground_truth_contexts,
        "generated_answer": generated_answer,
        "ground_truth_answer": ground_truth_answer,
        "contexts_from_response": [node.get_content() for node in response.source_nodes] # Contexts actually used by LLM
    })

    # Prepare data for Ragas
    ragas_data.append({
        "question": query,
        "contexts": retrieved_nodes_text, # Ragas expects retrieved contexts
        "answer": generated_answer,
        "ground_truth": ground_truth_answer
    })

print("Evaluation loop complete. Calculating metrics...")

# Convert results to DataFrame for easier processing
results_df = pd.DataFrame(results)

# --- Calculate Retrieval Metrics ---

k_val = 2 # Top-k for retrieval metrics

hit_rate = calculate_hit_rate(results_df["retrieved_nodes"].tolist(), results_df["ground_truth_contexts"].tolist(), k=k_val)
mrr = calculate_mrr(results_df["retrieved_nodes"].tolist(), results_df["ground_truth_contexts"].tolist())
recall_at_k = calculate_recall_at_k(results_df["retrieved_nodes"].tolist(), results_df["ground_truth_contexts"].tolist(), k=k_val)
precision_at_k = calculate_precision_at_k(results_df["retrieved_nodes"].tolist(), results_df["ground_truth_contexts"].tolist(), k=k_val)

# --- Calculate Generation & Overall RAG Metrics (using Ragas) ---

# Create Ragas Dataset
ragas_dataset = Dataset.from_list(ragas_data)

# Run Ragas evaluation
# Note: This step involves multiple LLM calls and can take some time.
print("\nRunning Ragas evaluation (this may take a few minutes due to LLM calls)...")
ragas_results = evaluate(ragas_dataset, metrics=ragas_metrics)

# --- Present Results ---

print("\n--- RAG Evaluation Summary ---")
print("\nRetrieval Metrics (Top-k = 2):")
print(f"  Hit Rate: {hit_rate:.4f}")
print(f"  Mean Reciprocal Rank (MRR): {mrr:.4f}")
print(f"  Recall@{k_val}: {recall_at_k:.4f}")
print(f"  Precision@{k_val}: {precision_at_k:.4f}")

print("\nGeneration & Overall RAG Metrics (Ragas):")
print(ragas_results.to_pandas().mean().to_string())

print("\n--- Detailed Ragas Results (First 5 samples) ---")
print(ragas_results.to_pandas().head().to_string())

print("\nEvaluation complete. Analyze the metrics to identify areas for improvement in your RAG pipeline.")

# Example of how to use LlamaIndex's built-in evaluators (can be integrated into the loop)
# For brevity, we'll just show one example here, but in a full solution, you'd run these for each query.
# first_query_result = results_df.iloc[0]
# print(f"\n--- LlamaIndex Built-in Evaluator Example (First Query) ---")
# print(f"Query: {first_query_result['query']}")
# print(f"Generated Answer: {first_query_result['generated_answer']}")

# faithfulness_score = faithfulness_evaluator.evaluate(
#     query=first_query_result['query'], 
#     response=first_query_result['generated_answer'], 
#     contexts=first_query_result['contexts_from_response']
# )
# print(f"Faithfulness (LlamaIndex): {faithfulness_score.score}")

# relevancy_score = relevancy_evaluator.evaluate(
#     query=first_query_result['query'], 
#     response=first_query_result['generated_answer']
# )
# print(f"Relevancy (LlamaIndex): {relevancy_score.score}")
